# Vanilla GCN on Elliptic++ (Local)

This notebook runs the full project workflow locally on the Elliptic++ files in `data/raw/`.

Set `DATASET = "transactions"` for the project’s Bitcoin transaction task, or `DATASET = "actors"` for wallet-address classification. The default transaction graph uses:

- `txs_features.csv` and `txs_classes.csv`
- `txs_edgelist.csv`

The actor option uses:

- `wallets_features.csv` and `wallets_classes.csv`
- `AddrAddr_edgelist.csv`

`AddrTx_edgelist.csv`, `TxAddr_edgelist.csv`, and `wallets_features_classes_combined.csv` remain in the dataset directory for other analyses and are not mixed into this homogeneous graph. The model architecture and training values remain unchanged; adjacency storage is sparse.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

requirements = PROJECT_ROOT / 'requirements.txt'
if requirements.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_local'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Dataset directory:', DATA_DIR)
print('Outputs directory:', OUTPUT_DIR)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
HIDDEN_DIM = 64
NUM_LAYERS = 2
DROPOUT = 0.5
EPOCHS = 200
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0005
TRAIN_RATIO = 0.6
VALIDATION_RATIO = 0.2
TEST_RATIO = 0.2
LOG_EVERY = 10

np.random.seed(SEED)
torch.manual_seed(SEED)
print('PyTorch:', torch.__version__)

## Load and normalize the selected Elliptic++ graph

Keep all listed CSV files together in `data/raw/`. Change only `DATASET` below when switching between transactions and actors.

In [ ]:
DATASET = 'transactions'  # change to 'actors' for wallet-address classification

DATASET_FILES = {
    'transactions': {
        'features': 'txs_features.csv',
        'classes': 'txs_classes.csv',
        'edges': 'txs_edgelist.csv',
        'expected_features': 183,
    },
    'actors': {
        'features': 'wallets_features.csv',
        'classes': 'wallets_classes.csv',
        'edges': 'AddrAddr_edgelist.csv',
        'expected_features': 56,
    },
}
if DATASET not in DATASET_FILES:
    raise ValueError(f'DATASET must be one of {list(DATASET_FILES)}')

selected = DATASET_FILES[DATASET]
FEATURES_PATH = DATA_DIR / selected['features']
CLASSES_PATH = DATA_DIR / selected['classes']
EDGES_PATH = DATA_DIR / selected['edges']
missing = [str(path) for path in (FEATURES_PATH, CLASSES_PATH, EDGES_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing files for DATASET=' + DATASET + ':\n' + '\n'.join('  - ' + path for path in missing)
    )

features_df = pd.read_csv(FEATURES_PATH, header=None)
node_ids = features_df.iloc[:, 0].astype(str).str.strip().to_numpy()
X_np = features_df.iloc[:, 1:].to_numpy(dtype=np.float32)
node_id_to_idx = {node_id: index for index, node_id in enumerate(node_ids)}
num_nodes = len(node_ids)

classes_df = pd.read_csv(CLASSES_PATH)
classes_df.columns = classes_df.columns.astype(str).str.strip()
class_id_column = classes_df.columns[0]
class_column = 'class' if 'class' in classes_df.columns else classes_df.columns[1]
raw_classes = classes_df.set_index(class_id_column)[class_column].astype(str).str.strip()
y_np = np.full(num_nodes, -1, dtype=np.int64)
for node_id, class_value in raw_classes.items():
    index = node_id_to_idx.get(str(node_id).strip())
    if index is not None and class_value == '1':
        y_np[index] = 0
    elif index is not None and class_value == '2':
        y_np[index] = 1

edges_df = pd.read_csv(EDGES_PATH)
edges_df.columns = edges_df.columns.astype(str).str.strip()
src_ids = edges_df.iloc[:, 0].astype(str).str.strip().to_numpy()
dst_ids = edges_df.iloc[:, 1].astype(str).str.strip().to_numpy()
valid = np.array(
    [(src_id in node_id_to_idx and dst_id in node_id_to_idx) for src_id, dst_id in zip(src_ids, dst_ids)],
    dtype=bool,
)
src = np.array([node_id_to_idx[node_id] for node_id in src_ids[valid]], dtype=np.int64)
dst = np.array([node_id_to_idx[node_id] for node_id in dst_ids[valid]], dtype=np.int64)

def make_sparse_normalized_adjacency(src, dst, num_nodes):
    nodes = np.arange(num_nodes, dtype=np.int64)
    all_src = np.concatenate([src, dst, nodes])
    all_dst = np.concatenate([dst, src, nodes])
    indices = torch.tensor(np.vstack([all_src, all_dst]), dtype=torch.long)
    values = torch.ones(indices.shape[1], dtype=torch.float32)
    adjacency = torch.sparse_coo_tensor(indices, values, (num_nodes, num_nodes)).coalesce()
    degree = torch.sparse.sum(adjacency, dim=1).to_dense()
    inv_sqrt = degree.clamp_min(1e-12).pow(-0.5)
    normalized = adjacency.values() * inv_sqrt[adjacency.indices()[0]] * inv_sqrt[adjacency.indices()[1]]
    return torch.sparse_coo_tensor(adjacency.indices(), normalized, adjacency.shape).coalesce()

def make_masks(labels):
    labelled = np.flatnonzero(labels >= 0)
    perm = np.random.default_rng(SEED).permutation(labelled)
    n_train = int(len(perm) * TRAIN_RATIO)
    n_val = int(len(perm) * VALIDATION_RATIO)
    train = np.zeros(len(labels), dtype=bool)
    validation = np.zeros(len(labels), dtype=bool)
    test = np.zeros(len(labels), dtype=bool)
    train[perm[:n_train]] = True
    validation[perm[n_train:n_train + n_val]] = True
    test[perm[n_train + n_val:]] = True
    return train, validation, test

train_mask, validation_mask, test_mask = make_masks(y_np)
y_safe = y_np.copy()
y_safe[y_safe < 0] = 0
A_tilde = make_sparse_normalized_adjacency(src, dst, num_nodes)
X = torch.from_numpy(X_np)
y = torch.from_numpy(y_safe)
train_mask = torch.from_numpy(train_mask)
validation_mask = torch.from_numpy(validation_mask)
test_mask = torch.from_numpy(test_mask)

if X_np.shape[1] != selected['expected_features']:
    raise ValueError(f'Expected {selected["expected_features"]} features for {DATASET}, found {X_np.shape[1]}')
print(f'Dataset: {DATASET}')
print(f'Nodes: {num_nodes:,}')
print(f'Features: {X_np.shape[1]}')
print(f'Sparse adjacency entries: {A_tilde._nnz():,}')
print(f'Labelled nodes: {(y_np >= 0).sum():,}')
print(f'Illicit: {(y_np == 0).sum():,} | Licit: {(y_np == 1).sum():,}')
print(f'Train/validation/test: {train_mask.sum():,}/{validation_mask.sum():,}/{test_mask.sum():,}')

## Train and evaluate the two-layer vanilla GCN

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, activation=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, output_dim))
        nn.init.xavier_uniform_(self.weight)
        self.activation = activation

    def forward(self, H, A_tilde):
        aggregated = torch.sparse.mm(A_tilde, H)
        output = aggregated @ self.weight
        return F.relu(output) if self.activation else output

class VanillaGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.layer1 = GCNLayer(input_dim, hidden_dim, activation=True)
        self.layer2 = GCNLayer(hidden_dim, num_classes, activation=False)

    def forward(self, X, A_tilde, return_hidden=False):
        hidden = self.layer1(X, A_tilde)
        hidden = F.dropout(hidden, p=DROPOUT, training=self.training)
        logits = self.layer2(hidden, A_tilde)
        return (logits, hidden) if return_hidden else logits

model = VanillaGCN(X.shape[1], HIDDEN_DIM, 2)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()
history = {'train_loss': [], 'validation_loss': [], 'train_accuracy': [], 'validation_accuracy': []}

def accuracy(logits, labels, mask):
    return (logits[mask].argmax(dim=1) == labels[mask]).float().mean().item() * 100

start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A_tilde)
    train_loss = criterion(logits[train_mask], y[train_mask])
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        evaluated = model(X, A_tilde)
        validation_loss = criterion(evaluated[validation_mask], y[validation_mask]).item()
        train_accuracy = accuracy(evaluated, y, train_mask)
        validation_accuracy = accuracy(evaluated, y, validation_mask)

    history['train_loss'].append(train_loss.item())
    history['validation_loss'].append(validation_loss)
    history['train_accuracy'].append(train_accuracy)
    history['validation_accuracy'].append(validation_accuracy)
    if epoch == 1 or epoch % LOG_EVERY == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | train loss {train_loss.item():.4f} | validation loss {validation_loss:.4f} | validation accuracy {validation_accuracy:.2f}%')

model.eval()
with torch.no_grad():
    final_logits, final_embeddings = model(X, A_tilde, return_hidden=True)
    predictions = final_logits.argmax(dim=1)
    test_accuracy = accuracy(final_logits, y, test_mask)

print(f'Training finished in {time.time() - start:.1f} seconds')
print(f'Test accuracy: {test_accuracy:.2f}%')

In [ ]:
checkpoint_path = OUTPUT_DIR / 'gcn_pytorch_checkpoint.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {'input_dim': X.shape[1], 'hidden_dim': HIDDEN_DIM, 'num_classes': 2, 'num_layers': NUM_LAYERS},
    'test_accuracy': test_accuracy,
}, checkpoint_path)
np.save(OUTPUT_DIR / 'node_embeddings.npy', final_embeddings.detach().numpy())
np.save(OUTPUT_DIR / 'predictions.npy', predictions.detach().numpy())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['validation_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(history['train_accuracy'], label='Train')
axes[1].plot(history['validation_accuracy'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_ylabel('Percent')
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()
print('Saved outputs to:', OUTPUT_DIR)
print('Checkpoint:', checkpoint_path)